# DICOM QC Review - Local Data

Batch quality control review of DICOM data from a local directory.

## Setup

1. Install the package: `uv pip install .` (from repo root)
2. Update `DATA_DIR` below to point to your DICOM data
3. Run all cells

## Features

- **Progress saving**: Automatically saves after every 10 series
- **Incremental processing**: Re-running skips already-processed series
- **Interactive viewer**: Click to navigate, scroll to change slices
- **DICOM header browser**: Search and filter tags

In [ ]:
# IMPORTANT: Run this cell first - must be before any matplotlib imports
%matplotlib widget

In [ ]:
from pathlib import Path
from dicom_qc import QuickCheck

# ============================================================
# CONFIGURE THESE PATHS
# ============================================================

# Directory containing DICOM files (searches recursively for *.dcm)
DATA_DIR = Path('/path/to/your/dicom/data')

# Where to save progress and reports (defaults to current directory)
OUTPUT_DIR = Path('.')

# ============================================================

# Save file stores discovery + processing progress
SAVE_FILE = OUTPUT_DIR / 'qc_state.pkl'

# Load existing progress or start fresh
if SAVE_FILE.exists():
    print(f"Loading saved state from {SAVE_FILE}")
    qc = QuickCheck.from_save(SAVE_FILE)
else:
    print(f"Starting fresh - scanning {DATA_DIR}")
    qc = QuickCheck(DATA_DIR)

qc._save_path = SAVE_FILE  # Enable auto-save during processing

In [ ]:
# Discover DICOM files
# - Recursively scans DATA_DIR for *.dcm files
# - Builds patient/study/series hierarchy from DICOM headers
# - Preserves previously processed series

qc.discover()
print(f"Found {len(qc.patients)} patients, {len(qc.get_all_series())} series")

In [ ]:
# Process series (run QC checks, generate thumbnails)
# - Skips already-processed series
# - Auto-saves every 10 series
# - Shows live progress with thumbnails

qc.process_all_interactive()

In [ ]:
# Generate HTML report and save final state
report_path = OUTPUT_DIR / 'qc_report.html'
qc.generate_html_report(report_path)
qc.save()
print(f"Report saved to: {report_path}")

In [ ]:
# Interactive review dashboard
# - Click "View" to see 3-pane viewer
# - Click "Tags" to browse DICOM headers
# - Use filters to find specific series

qc.display()

## Tips

### Re-process specific series
```python
qc.process_all_interactive(reprocess=True)  # Re-run all
qc.process_all_interactive(retry_errors=True)  # Only retry errors
```

### Start fresh
```python
# Delete save file and restart kernel, or:
qc = QuickCheck(DATA_DIR)
qc._save_path = SAVE_FILE
```

### Filter the report
Use the dropdown filters above the series list to filter by:
- Status (Pass, Warning, Fail, etc.)
- Subject
- Session
- Series number
- Description text